In [1]:
import numpy as np
import pandas as pd
import sklearn

import itertools
import ast
import warnings
import pickle
import random
import os
import time
from datetime import datetime
from itertools import combinations
import copy
import gc

from sklearn.metrics import accuracy_score, f1_score, accuracy_score, make_scorer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold, StratifiedKFold
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
import sys
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError
from sklearn.preprocessing import LabelEncoder
from mlxtend.classifier import EnsembleVoteClassifier
import copy
from bayes_opt import BayesianOptimization
from functools import partial
from sklearn.base import clone
from sklearn.naive_bayes import GaussianNB
import scipy
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore") # No quiero ver los warnings en entrenamiento

print('The scikit-learn version is {}.'.format(sklearn.__version__))
print('The numpy version is {}.'.format(np.__version__))
print('The pandas version is {}.'.format(pd.__version__))
print('The Python version is {}.'.format(sys.version))

from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier, OutputCodeClassifier


The scikit-learn version is 1.7.2.
The numpy version is 2.0.0.
The pandas version is 2.2.2.
The Python version is 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:12:24) [GCC 11.2.0].


In [2]:
import traceback
#import datetime
from pathlib import Path

def log_exception(exception, name, run, file_path="error_log.txt"):
    """
    Registra una excepción en un archivo de texto.
    
    Args:
        exception: La excepción capturada
        file_path: Ruta del archivo donde se guardará el error
    """
    # Obtener la fecha y hora actual
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Obtener el traceback completo
    error_traceback = traceback.format_exc()
    
    # Crear el mensaje de error con formato
    error_message = f"""
    ===== ERROR {timestamp} =====
    Type: {type(exception).__name__}
    Dataset: {name}
    Run: {str(run)}
    Message: {str(exception)}
    Traceback:
    {error_traceback}
    =============================
    """
    
    # Asegurarse de que el directorio del archivo existe
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)
    
    # Escribir en el archivo en modo append
    with open(file_path, "a", encoding="utf-8") as file:
        file.write(error_message)
    
    print(f"Exception '{file_path}'")

In [3]:
from series_parallel import *

In [4]:
FOLDER_DATASETS = "datasets_multiclass/"

def get_model_names(list_models):
    return [str(i).split('(')[0] for i in list_models]

def get_name_from_model(model):
    return str(model).split('(')[0]

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

def create_seed_models(number_trained_models = 6, random_state=123):
    seed_everything(random_state)
    KNN = KNeighborsClassifier()
    SVClassifier = SVC(probability=True, random_state = random_state)
    RIDGE = RidgeClassifier(random_state = random_state) # This will be a CalibratedClassifierCV(base_estimator=ridge, method='sigmoid', cv=5)
    # GNB = GaussianNB()
    DT = DecisionTreeClassifier(random_state = random_state)
    MLP = MLPClassifier(random_state=random_state)
    logreg = LogisticRegression(random_state=random_state)
    
    list = [KNN, SVClassifier, RIDGE, DT, MLP, logreg]
    #Lanzar un warning si number_models<2 o >6 en este momento.
    return list[:number_trained_models]    



def perform_bayesian_optimization(base_model, random_state, train_models_X, train_models_Y, train_ensemble_X, train_ensemble_Y, metric, init_points = 2, n_iter=10, verbose=True):  
    # Function for Bayesian Optimization
    def model_evaluation(model, train_models_X, train_models_Y, train_ensemble_X, train_ensemble_Y, metric, **params):
        """
        Evaluates a model based on its index in the list and the provided hyperparameters.
        - model_index: index of the model in the list generated by `create_seed_models`.
        - params: specific hyperparameters for the model.
        """

        # Ajustamos los hiperparámetros al modelo
        if isinstance(model, KNeighborsClassifier):
            model.set_params(
                n_neighbors=int(params["n_neighbors"]),
                p=int(params["p"])
            )
        elif isinstance(model, SVC):
            model.set_params(
                C=10 ** params["C"],
                gamma=10 ** params.get("gamma", -4),
                kernel="rbf"
            )
        elif isinstance(model, GaussianNB):
            model.set_params(var_smoothing=10 ** params["var_smoothing"])
        elif isinstance(model, DecisionTreeClassifier):
            model.set_params(
                max_depth=int(params["max_depth"]),
                min_samples_split=int(params["min_samples_split"])
            )
        elif isinstance(model, MLPClassifier):
            model.set_params(
                hidden_layer_sizes=(int(params["hidden_layer_sizes"]),),
                alpha=10 ** params["alpha"],
                solver="lbfgs",
                activation="logistic",
                max_iter=10000
            )
        elif isinstance(model, LogisticRegression):
            model.set_params(
                C=10 ** params["C"],
                max_iter=int(params.get("max_iter", 1000))
            )
        elif isinstance(model, RidgeClassifier):
            model.set_params(
                alpha=10 ** params["alpha"]
            )
            model = CalibratedClassifierCV(estimator=model, method="sigmoid", cv=5)            
        else:
            raise ValueError(f"Modelo no reconocido: {model}")

       # Training in train set
        model.fit(train_models_X, train_models_Y)

        # Prediction in val (train_ensemble)
        y_pred = model.predict(train_ensemble_X)

        return metric(train_ensemble_Y, y_pred)

    search_space_Bayesopt = {
    "KNeighborsClassifier": { 
        "n_neighbors": (1, 50),
        "p": (1, 3)
    },
    "SVC": {  # SVC
        "C": (-4, 4),
        "gamma": (-4, 4)
    },
    "GaussianNB": {
        "var_smoothing": (-12, -5)  # 10^-12 a 10^-5
    },
    "DecisionTreeClassifier": {
        "max_depth": (2, 30),
        "min_samples_split": (2, 25)
    },
    "MLPClassifier": {
        "hidden_layer_sizes": (1, 100), # Pongo 100 (estaba en 30).
        "alpha": (-4, 4)
    },
    "LogisticRegression": {
        "C": (-4, 4),
        "max_iter": (1000, 1000)  # Valor fijo
    },
    "RidgeClassifier": {
        "alpha": (-4, 4),  # busca entre 10^-3 y 10^1
    }
    }
    
    model_name = get_name_from_model(base_model)
    param_bounds = search_space_Bayesopt[model_name]
    optimizer = BayesianOptimization(
        f=lambda **params: model_evaluation(
            model=clone(base_model),  # Clonar el modelo para cada evaluación
            train_models_X = train_models_X,
            train_models_Y = train_models_Y,
            train_ensemble_X = train_ensemble_X, 
            train_ensemble_Y = train_ensemble_Y, 
            metric=metric, 
            **params
        ),
        pbounds=param_bounds,
        allow_duplicate_points=True,
        random_state=random_state,
        verbose = False
    )
    if verbose:
        print("Optimizing the base model", model_name)
    optimizer.maximize(init_points=init_points, n_iter=n_iter)

    best_params = optimizer.max["params"]

    best_model = clone(base_model)

    if isinstance(best_model, KNeighborsClassifier):
        best_model.set_params(
            n_neighbors=int(best_params["n_neighbors"]),
            p=int(best_params["p"])
        )
    elif isinstance(best_model, SVC):
        best_model.set_params(
            C=10 ** best_params["C"],
            gamma=10 ** best_params.get("gamma", -4),
            kernel="rbf"
        )
    elif isinstance(best_model, GaussianNB):
        best_model.set_params(var_smoothing=10 ** best_params["var_smoothing"])
    elif isinstance(best_model, DecisionTreeClassifier):
        best_model.set_params(
            max_depth=int(best_params["max_depth"]),
            min_samples_split=int(best_params["min_samples_split"])
        )
    elif isinstance(best_model, MLPClassifier):
        best_model.set_params(
            hidden_layer_sizes=(int(best_params["hidden_layer_sizes"]),),
            alpha=10 ** best_params["alpha"],
            solver="lbfgs",
            activation="logistic",
            max_iter=10000
        )
    elif isinstance(best_model, LogisticRegression):
        best_model.set_params(
            C=10 ** best_params["C"],
            max_iter=int(best_params.get("max_iter", 1000))
        )
    elif isinstance(best_model, RidgeClassifier):
        best_model.set_params(
            alpha=10 ** best_params["alpha"]
        )
        best_model = CalibratedClassifierCV(estimator=best_model, method="sigmoid", cv=5)      
    else:
        raise ValueError(f"Modelo no reconocido: {type(best_model)}")

    best_model.fit(train_models_X, train_models_Y)  # Entrenar en todo X_train después de optimización

    return best_model

In [5]:
class dataset():
    def __init__(self, name_ds, metric = None, random_state = 123, range_number_models_ensemble = range(3,7)):
        self.name_file = name_ds
        
        self.train_models_X, self.train_models_Y, \
        self.train_ensemble_X, self.train_ensemble_Y, self.test_X, self.test_Y = [None]*6
        
        self.base_models_optimized_sorted_by_val = None # Aquí estarán los modelos base entrenados, optimizados con BO y ordenados (de mayor a menor) por score en VAL.                
        
        self.voting_ensemble_hard = None
        self.voting_ensemble_soft = None
        
        self.stacking_ensemble = None
        
        self.bayesian_init_points = 2 # 10
        self.bayesian_n_iter = 10 # 50
        
        self.random_forest = None
        self.xgboost = None        
        
        self.voting_ensemble_hard_topN = None
        self.voting_ensemble_soft_topN = None
        self.stacking_ensemble_topN = None
        
        self.random_state = random_state 
        self.range_number_models_ensemble = range_number_models_ensemble
        
        if metric is None:
            self.metric = accuracy_score
        else:
            self.metric = metric
        self.dict_results = dict(name_dataset = name_ds, random_state = random_state, metric = self.metric)
        
        
        # self._model_names = [str(i).split('(')[0] for i in create_seed_models(random_state = self.random_state)]
        


    def load_dataset(self): #60% train, 20% val, 20% test. Stratified.
        # Cargamos el dataset
        X = pd.read_csv(self.name_file)
        y = X['target_end']
                
        X.drop(columns='target_end', inplace=True)
        input_names = X.columns

        # OJO: Añado esto porque ahora es necesario con XGboost (https://stackoverflow.com/questions/71996617/invalid-classes-inferred-from-unique-values-of-y-expected-0-1-2-3-4-5-got)
        le = LabelEncoder()
        y = le.fit_transform(y)
        
        # Al hacer lo del labelEnconder tengo que quitar el values
        # self.train_models_X, self.test_X, self.train_models_Y, self.test_Y = train_test_split(X.values, y.values, test_size=0.2, stratify = y.values, random_state=self.random_state)
        self.train_models_X, self.test_X, \
        self.train_models_Y, self.test_Y = train_test_split(X.values, y, test_size=0.2, stratify = y, random_state=self.random_state)     
        
        
        self.train_models_X, self.train_ensemble_X,\
        self.train_models_Y, self.train_ensemble_Y = train_test_split(self.train_models_X,  self.train_models_Y, 
                                                                      test_size=0.25, 
                                                                      stratify = self.train_models_Y,
                                                                      random_state = self.random_state)       

    def compute_metrics(self, model_name, model, train=False, val=False, test=True, mode=None):
        results = {}               
        if model is None: # Sanity check
            results[model_name + "_test_macro"] = 0.0
            results[model_name + "_test_micro"] = 0.0
            results[model_name + "_test_weighted"] = 0.0
        else:
            if test:
                preds = model.predict_multiclass(self.test_X) if mode=="multiclass" else model.predict(self.test_X)
                results[model_name + "_test_macro"] = f1_score(self.test_Y, preds, average='macro')
                results[model_name + "_test_micro"] = f1_score(self.test_Y, preds, average='micro')
                results[model_name + "_test_weighted"] = f1_score(self.test_Y, preds, average='weighted')
                print(model_name + "_test_macro", results[model_name + "_test_macro"]) # Output only the _test_macro score.
            if train:
                preds = model.predict_multiclass(self.train_models_X) if mode=="multiclass" else model.predict(self.train_models_X)
                results[model_name + "_train_macro"] = f1_score(self.train_models_Y, preds, average='macro')
                # results[model_name + "_test_micro"] = f1_score(self.train_models_Y, preds, average='micro')
                # results[model_name + "_test_weighted"] = f1_score(self.train_models_Y, preds, average='weighted')
            if val:           
                preds = model.predict_multiclass(self.train_ensemble_X) if mode=="multiclass" else model.predict(self.train_ensemble_X)
                results[model_name + "_val_macro"] = f1_score(self.train_ensemble_Y, preds, average='macro')
        self.dict_results.update(results)
                      
        
        
        
    def train_base_models(self):
        base_models = {} 
        seed_models = create_seed_models(random_state = self.random_state)
        results = {}        
        model_names = get_model_names(seed_models)
        
        # for model_index, param_bounds in search_space_Bayesopt.items():
        #     base_model = seed_models[model_index]
        #     model_name = model_names[model_index]
        for base_model, model_name in zip(seed_models, model_names):
            best_model = perform_bayesian_optimization(base_model, self.random_state, 
                                                       self.train_models_X, self.train_models_Y, self.train_ensemble_X, 
                                                       self.train_ensemble_Y, self.metric, 
                                                       init_points = self.bayesian_init_points, n_iter=self.bayesian_n_iter)
            base_models[model_name] = best_model  # Guardar el modelo entrenado.
            results[model_name + "_optimized_train_macro"] = self.metric(self.train_models_Y, best_model.predict(self.train_models_X)) # TRAIN
            results[model_name + "_optimized_val_macro"] = self.metric(self.train_ensemble_Y, best_model.predict(self.train_ensemble_X)) # VAL            
            self.compute_metrics(model_name + "_optimized", best_model)   # Esto calcula métrica en test         
        
        wanted_keys = [model_name + "_optimized_val_macro" for model_name in model_names]
        dict_scores_val = dict((k.removesuffix("_optimized_val_macro"), results[k]) for k in wanted_keys if k in results) 
        dict_scores_val = dict(sorted(dict_scores_val.items(), key=lambda item: item[1], reverse=True)) 

        best_models_names = list(dict_scores_val.keys()) 
        self.base_models_optimized_sorted_by_val = [base_models.get(key) for key in best_models_names] # Los guardo ordenados por mejor score en VAL
        # self._model_names_optimized_sorted_by_val = [str(i).split('(')[0] for i in self.base_models_optimized_sorted_by_val] # Aquí están los nombres de los modelos ordenados.
                
        results["params_base_models"] = [str(model) for model in self.base_models_optimized_sorted_by_val] # Guardamos como string
        self.dict_results.update(results)
                
            
    def train_voting_ensemble(self, voting="hard", only_top_n=False):
        label_prefix = "topN_" if only_top_n else ""
        list_best_voting_models = []             
        for number_models_ensemble in self.range_number_models_ensemble:
            tic = time.time()
            if only_top_n:
                # Directamente los N mejores modelos según validación
                model_combinations = [tuple(self.base_models_optimized_sorted_by_val[:number_models_ensemble])]
            else:
                # Get all combinations of number_models_ensemble models from the list of 6
                model_combinations = list(combinations(self.base_models_optimized_sorted_by_val, number_models_ensemble))
            best_metric = -1
            best_voting = None
            # Print the combinations
            for combo in model_combinations:
                models = list(combo)

                estimators = [(get_name_from_model(model), model) for model in models]
              #  tic = time.time()          
            
                voting_ensemble = EnsembleVoteClassifier(clfs=models, voting=voting, fit_base_estimators=False).fit(self.train_models_X,self.train_models_Y)       

                metric = self.metric(self.train_ensemble_Y, voting_ensemble.predict(self.train_ensemble_X)) # Comprobar VAL
                if metric > best_metric:
                    best_metric = metric
                    best_voting = voting_ensemble

            list_best_voting_models.append(best_voting)
            time_elapsed = time.time() - tic
            self.dict_results.update({"time_" + label_prefix + "voting_" + voting + "_" + str(number_models_ensemble) : time_elapsed})
        
        # Y ahora evaluamos todos en TRAIN, VAL y TEST (el mejor voting será el que tenga mejor VAL).
        for voting_model, number_models in zip(list_best_voting_models, self.range_number_models_ensemble):     
            self.compute_metrics(label_prefix + "voting_"+voting + "_" + str(number_models), voting_model, train=True, val=True, test=True)                         
            
        if voting == "hard":
            if only_top_n:
                self.voting_ensemble_hard_topN = list_best_voting_models
            else:
                self.voting_ensemble_hard = list_best_voting_models
        else:
            if only_top_n:
                self.voting_ensemble_soft_topN = list_best_voting_models
            else:
                self.voting_ensemble_soft = list_best_voting_models
         
    
    def train_series_parallel_multiclass(self, only_top_n=False):
        label_prefix = "topN_" if only_top_n else ""
        list_best_sp_multiclass_models = []             
        for number_models_ensemble in self.range_number_models_ensemble:
            tic = time.time()
            if only_top_n:
                # Directamente los N mejores modelos según validación
                model_combinations = [tuple(self.base_models_optimized_sorted_by_val[:number_models_ensemble])]
            else:
                # Get all combinations of number_models_ensemble models from the list of 6
                model_combinations = list(combinations(self.base_models_optimized_sorted_by_val, number_models_ensemble))
            best_metric = -1
            best_sp_multiclass = None
            # Print the combinations
            for combo in model_combinations:
                models = list(combo)
                estimators = [(get_name_from_model(model), model) for model in models]
              #  tic = time.time()          
            
                sp_multiclass_ensemble = SeriesParallelEnsemble(estimators, mode="multiclass").fit(self.train_ensemble_X, self.train_ensemble_Y, 
                                                                                                   metric=self.metric, n_jobs=1)      

                metric = self.metric(self.train_ensemble_Y, sp_multiclass_ensemble.predict_multiclass(self.train_ensemble_X)) # Comprobar VAL
                if metric > best_metric:
                    best_metric = metric
                    best_sp_multiclass = copy.deepcopy(sp_multiclass_ensemble)

            list_best_sp_multiclass_models.append(copy.deepcopy(best_sp_multiclass))
            time_elapsed = time.time() - tic
            self.dict_results.update({"time_" + label_prefix + "sp_multiclass_" + str(number_models_ensemble) : time_elapsed})
        
        # Y ahora evaluamos todos en TRAIN, VAL y TEST (el mejor modelo será el que tenga mejor VAL).
        for sp_model, number_models in zip(list_best_sp_multiclass_models, self.range_number_models_ensemble):     
            self.compute_metrics(label_prefix + "sp_multiclass_" + str(number_models), sp_model, 
                     train=True, val=True, test=True, mode="multiclass")   
            
            
            
    def _evaluate_stacking(self, base_models, C_log):
        """
        Evalúa el modelo de stacking con validación cruzada en VAL, optimizando el parámetro C (en logaritmo).
        """
        C = 10 ** C_log
    
        stack_model = StackingClassifier(
            estimators=base_models, 
            final_estimator=LogisticRegression(C=C, max_iter=10000, random_state=self.random_state), 
            cv="prefit" 
        )
        

        cv_scores_model = []
        skf = StratifiedKFold(n_splits=5)
        for i, (train_index, test_index) in enumerate(skf.split(self.train_ensemble_X, self.train_ensemble_Y)):
            stack_model.fit(self.train_ensemble_X[train_index], self.train_ensemble_Y[train_index])                                      
            cv_scores_model.append(self.metric(self.train_ensemble_Y[test_index], stack_model.predict(self.train_ensemble_X[test_index])))
                        
        return np.mean(cv_scores_model)


    def _optimize_stacking(self, models):
        """
        Optimiza el parámetro C del meta-modelo en escala logarítmica usando optimización bayesiana.
    
        Parameters:
        - models: lista de modelos base preajustados.
        - X_val, y_val: conjunto de validación.
        
        Returns:
        - Mejor valor de C encontrado y la precisión promedio.
        """
        pbounds = {"C_log": (-4, 4)}  # Log10(C) en [-4, 4] => C en [10^-4, 10^4]
        
        optimizer = BayesianOptimization(
            f=lambda C_log: self._evaluate_stacking(models, C_log),
            pbounds=pbounds,
            random_state=self.random_state,
            verbose = False
        )

        optimizer.maximize(init_points=self.bayesian_init_points, n_iter=self.bayesian_n_iter)
        
        best_C_log = optimizer.max["params"]["C_log"]
        best_C = 10 ** best_C_log  # Convertir de log a escala normal        
        return best_C, optimizer.max["target"]
    
    # Entrenamos el stacking con gridsearch para elegir bien el C
    def train_stacking_ensemble(self, only_top_n=False):    
        label_prefix = "topN_" if only_top_n else ""
        stacking_list = []
        for number_models_ensemble in self.range_number_models_ensemble:
            tic = time.time()
            if only_top_n:
                # Directamente los N mejores modelos según validación
                model_combinations = [tuple(self.base_models_optimized_sorted_by_val[:number_models_ensemble])]
            else:
                # Get all combinations of number_models_ensemble models from the list of 6
                model_combinations = list(combinations(self.base_models_optimized_sorted_by_val, number_models_ensemble))
            best_metric = -1
            best_stacking = None
            
            # Print the combinations
            for combo in model_combinations:
                models = list(combo)
                estimators = [(get_name_from_model(model), model) for model in models]
              #  tic = time.time()                         
                # Optimizar C para el StackingClassifier
                best_C, best_score = self._optimize_stacking(estimators)

                # Entrenar el modelo final con el mejor C encontrado
                final_stacking_model = StackingClassifier(
                    estimators=estimators, 
                    final_estimator=LogisticRegression(C=best_C, max_iter=10000, random_state=self.random_state), 
                    cv="prefit"  # Usar modelos base ya entrenados
                )
                # Entrenamos el meta-modelo en todo VAL
                final_stacking_model = final_stacking_model.fit(self.train_ensemble_X, self.train_ensemble_Y)
                metric = self.metric(self.train_ensemble_Y, final_stacking_model.predict(self.train_ensemble_X)) # Comprobar VAL
                if metric > best_metric:
                    best_metric = metric
                    best_stacking = copy.deepcopy(final_stacking_model)
                
            stacking_list.append(copy.deepcopy(best_stacking))
            time_elapsed = time.time() - tic
            self.dict_results.update({"time_" + label_prefix + "stacking_" + str(number_models_ensemble) : time_elapsed})
        
        for stacking_model, number_models in zip(stacking_list, self.range_number_models_ensemble):            
            self.compute_metrics(label_prefix + "stacking_" + str(number_models), stacking_model, train=True, val=True, test=True)   
            # Calculo los scores correspondientes de cada modelo:
        
        if only_top_n:
            self.stacking_ensemble_topN = stacking_list
        else:
            self.stacking_ensemble = stacking_list                        
    
    def train_random_forest(self):
        """
        Optimiza hiperparámetros de un RandomForestClassifier usando optimización bayesiana,
        entrenando en TRAIN y seleccionando el mejor modelo en VAL.

        Returns:
        - best_model: El mejor modelo ajustado.
        - best_params: Los mejores hiperparámetros encontrados.
        - best_score: El puntaje de precisión en el conjunto de validación.
        """
        # Función objetivo para optimización bayesiana
        def evaluate_rf(n_estimators, max_depth, min_samples_split):
            """
            Entrena y evalúa un RandomForestClassifier con los hiperparámetros dados.
            
            Parameters:
            - n_estimators: Número de árboles.
            - max_depth: Profundidad máxima.
            - min_samples_split: Número mínimo de muestras para dividir un nodo.

            Returns:
            - accuracy: Precisión en el conjunto de validación.
            """
            # Convertir hiperparámetros a enteros
            n_estimators = int(n_estimators)
            max_depth = int(max_depth)
            min_samples_split = int(min_samples_split)
            
            # Crear y entrenar el modelo
            model = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                random_state=self.random_state,
                n_jobs = -1
            )
            model.fit(self.train_models_X, self.train_models_Y)
            
            # Evaluar en el conjunto de validación
            predictions = model.predict(self.train_ensemble_X)
            # score = f1_score(self.train_ensemble_Y, predictions, average='macro')
            score = self.metric(self.train_ensemble_Y, predictions)
            
            return score

        tic = time.time()          
        # Rango de hiperparámetros para la optimización
        pbounds = {
            "n_estimators": (10, 1000),  # Número de árboles entre 10 y 1000
            "max_depth": (2, 20),       # Profundidad máxima entre 2 y 20
            "min_samples_split": (2, 25)  # Mínimas muestras para dividir entre 2 y 25
        }
        
        # Configuración de la optimización bayesiana
        optimizer = BayesianOptimization(
            f=evaluate_rf,    # Función objetivo
            pbounds=pbounds,  # Rango de hiperparámetros
            random_state=self.random_state,   # Para reproducibilidad
            verbose = False
        )
        
        # Maximizar f1_score
        optimizer.maximize(init_points=self.bayesian_init_points, n_iter=self.bayesian_n_iter)
        
        # Obtener los mejores parámetros
        best_params = optimizer.max["params"]
        best_params = {
            "n_estimators": int(best_params["n_estimators"]),
            "max_depth": int(best_params["max_depth"]),
            "min_samples_split": int(best_params["min_samples_split"])
        }
        
        # Entrenar el mejor modelo con los mejores hiperparámetros
        best_model = RandomForestClassifier(
            **best_params,
            random_state=self.random_state
        )
        best_model.fit(self.train_models_X, self.train_models_Y)                        
        self.random_forest = best_model

        time_elapsed = time.time() - tic           
        self.dict_results.update({"time_random_forest" : time_elapsed,
                                 "random_forest_params" : str(self.random_forest)})
        
        # Calculo los scores correspondientes del modelo:
        self.compute_metrics("random_forest", self.random_forest)
        


    def train_xgboost(self):
        """
        Optimiza hiperparámetros de un XGBoostClassifier usando optimización bayesiana,
        entrenando en TRAIN y seleccionando el mejor modelo en VAL.

        Returns:
        - best_model: El mejor modelo ajustado.
        - best_params: Los mejores hiperparámetros encontrados.
        - best_score: El puntaje de precisión en el conjunto de validación.
        """
        # Función objetivo para optimización bayesiana
        def evaluate_xgb(n_estimators, max_depth, subsample, colsample_bytree):
            """
            Entrena y evalúa un XGBoostClassifier con los hiperparámetros dados.
            
            Parameters:
            - n_estimators: Número de árboles.
            - max_depth: Profundidad máxima.
            - subsample: Fracción de muestras usadas para entrenar cada árbol.
            - colsample_bytree: Fracción de características usadas para entrenar cada árbol.

            Returns:
            - accuracy: Precisión en el conjunto de validación.
            """
            # Convertir hiperparámetros a sus tipos correctos
            n_estimators = int(n_estimators)
            max_depth = int(max_depth)
            
            # Crear y entrenar el modelo
            model = XGBClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                use_label_encoder=False,
      #          eval_metric='logloss',
                random_state=self.random_state,
                n_jobs=-1 # OJO: añado esto.
            )
            model.fit(self.train_models_X, self.train_models_Y)
            
            # Evaluar en el conjunto de validación            
            predictions = model.predict(self.train_ensemble_X) 
            score = self.metric(self.train_ensemble_Y, predictions)
            return score
            
        tic = time.time()     
        # Rango de hiperparámetros para la optimización
        pbounds = {
            "n_estimators": (10, 1000),         # Número de árboles entre 10 y 1000
            "max_depth": (2, 12),              # Profundidad máxima entre 2 y 12
            "subsample": (0.1, 1.0),           # Submuestreo entre 0.1 y 1.0
            "colsample_bytree": (0.1, 1.0)     # Fracción de características entre 0.1 y 1.0
        }
        
        # Configuración de la optimización bayesiana
        optimizer = BayesianOptimization(
            f=evaluate_xgb,    # Función objetivo
            pbounds=pbounds,   # Rango de hiperparámetros
            random_state=self.random_state,    # Para reproducibilidad
            verbose = False
        )
        
        # Maximizar f1_score
        optimizer.maximize(init_points=self.bayesian_init_points, n_iter=self.bayesian_n_iter)
        
        # Obtener los mejores parámetros
        best_params = optimizer.max["params"]
        best_params = {
            "n_estimators": int(best_params["n_estimators"]),
            "max_depth": int(best_params["max_depth"]),
            "subsample": best_params["subsample"],
            "colsample_bytree": best_params["colsample_bytree"]
        }
        
        # Entrenar el mejor modelo con los mejores hiperparámetros
        best_model = XGBClassifier(
            **best_params,
            use_label_encoder=False,
            #eval_metric='logloss',
            random_state=self.random_state
        )
        self.xgboost = best_model.fit(self.train_models_X, self.train_models_Y)
        
        time_elapsed = time.time() - tic           
        self.dict_results.update({"time_xgboost" : time_elapsed, 
                                 "xgboost_params" : str(self.xgboost)})
        
        # Calculo los scores correspondientes del modelo:
        self.compute_metrics("xgboost", self.xgboost)
    

    
    def cleanup(self):
        """Libera memoria de objetos grandes"""
        # Limpiar datos
        self.train_models_X = None
        self.train_models_Y = None
        self.train_ensemble_X = None
        self.train_ensemble_Y = None
        self.test_X = None
        self.test_Y = None

        # Limpiar modelos
        self.base_models_optimized_sorted_by_val = None
        self.voting_ensemble_hard = None
        self.voting_ensemble_soft = None
        self.voting_ensemble_hard_topN = None
        self.voting_ensemble_soft_topN = None
        self.stacking_ensemble_topN = None
        self.random_forest = None
        self.xgboost = None    
        
        # Forzar garbage collection
        gc.collect()
    

    def run(self, verbose = 0):
        print("------ RUNNING DATASET", self.name_file, "------")
        self.load_dataset()

        self.train_base_models()
        
        # Series-Parallel: búsqueda exhaustiva (todas las combinaciones)
        self.train_series_parallel_multiclass()
        # Series-Parallel: solo los N mejores modelos (sin explorar combinaciones)
        self.train_series_parallel_multiclass(only_top_n=True)

                        
        # Entrenamos los voting: búsqueda exhaustiva
        self.train_voting_ensemble(voting="hard")
        self.train_voting_ensemble(voting="soft")
        # Voting: solo los N mejores modelos
        self.train_voting_ensemble(voting="hard", only_top_n=True)
        self.train_voting_ensemble(voting="soft", only_top_n=True)
                                                  
        
        # Entrenamos el stacking (ajustando el C con BO). El tiempo de cada modelo se calcula dentro de la propia función.
        self.train_stacking_ensemble()
        self.train_stacking_ensemble(only_top_n=True)
        
        # Entrenamos el random forest con parámetros optimizados con BO
        self.train_random_forest()
        
        # Entrenamos el xgboost con parámetros optimizados con BO.
        self.train_xgboost()
        
                

In [6]:
import glob

warnings.filterwarnings("ignore") # No quiero ver los warnings en entrenamiento
# df = pd.read_csv("mat_mc.csv")

results = []
now = datetime.now()
NAME_FILE_CSV = 'results_multiclass' + now.strftime("%Y_%m_%d_%H_%M_%S") + ".csv"
NRUNS = 10
RANGE_NUMBER_MODELS_ENSEMBLE = range(3,7)

list_datasets = glob.glob("datasets_multiclass/*.csv")
print(list_datasets)
for run in range(NRUNS):
    for i in list_datasets:
        try:
            d = dataset(i, metric = lambda y_true, y_pred: f1_score(y_true,y_pred, average='macro'), 
                        random_state = 123 * (run+1), range_number_models_ensemble = RANGE_NUMBER_MODELS_ENSEMBLE)
            d.run(verbose = True)    
            # Resultados:
            d.dict_results.update({"run" : run})
            results.append(copy.deepcopy(d.dict_results))   
            d.cleanup()
            del d      
            gc.collect()
            results_df = pd.DataFrame(results)
            results_df.to_csv(NAME_FILE_CSV, index = False)
        except Exception as e:
            print("Failed dataset", d.name_file)
            print(e) # If somehow a dataset fails, then pass  
            log_exception(e,d.name_file, run, "exceptions.txt")
            pass 

['datasets_multiclass/splice.csv', 'datasets_multiclass/first.csv', 'datasets_multiclass/eye.csv', 'datasets_multiclass/wall.csv', 'datasets_multiclass/allbp.csv', 'datasets_multiclass/jungle.csv', 'datasets_multiclass/gas.csv', 'datasets_multiclass/cjs.csv', 'datasets_multiclass/GesturePhaseSegmentationProcessed.csv', 'datasets_multiclass/page.csv', 'datasets_multiclass/Indian.csv', 'datasets_multiclass/microaggregation2.csv', 'datasets_multiclass/JapaneseVowels.csv', 'datasets_multiclass/led24.csv', 'datasets_multiclass/texture.csv', 'datasets_multiclass/dna.csv', 'datasets_multiclass/segment.csv', 'datasets_multiclass/USPS.csv', 'datasets_multiclass/thyroid.csv', 'datasets_multiclass/autoUniv.csv']
------ RUNNING DATASET datasets_multiclass/splice.csv ------
Optimizing the base model KNeighborsClassifier
KNeighborsClassifier_optimized_test_macro 0.8851521251776976
Optimizing the base model SVC
SVC_optimized_test_macro 0.9512186996532045
Optimizing the base model RidgeClassifier
Ridg